# 5. Feature engineering and feature selection

## Small idea: features should answer the prediction question

Feature engineering converts raw observations into measurements that may express a
useful pattern. Feature selection removes irrelevant, redundant, unstable, forbidden,
or excessively costly features. Both steps must respect prediction time and the split.

**Learning goals**

- construct target-independent text and metadata features;
- avoid identifiers and post-outcome variables;
- remove constant features;
- fit supervised selectors on training data only;
- understand why domain reasoning comes before automatic selection.

In [ ]:
import re
from functools import partial
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, VarianceThreshold, mutual_info_classif

## 1. Build transparent, target-independent features

In [ ]:
responses = pd.DataFrame({
    "response_id": ["R1", "R2", "R3", "R4", "R5", "R6"],
    "text": [
        "من فارسی را دوست دارم.",
        "واقعاً؟ این تمرین سخت بود!",
        "سلام 😊😊",
        "من امروز به دانشگاه می‌روم.",
        "کتاب جدید خریدم",
        "چرا این‌قدر سخت است؟؟",
    ],
    "response_seconds": [75, 180, 30, 120, 90, 210],
    "annotated_errors": [1, 5, 0, 2, 1, 6],
    "needs_review": [0, 1, 0, 0, 0, 1],
})

def add_surface_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["character_count"] = result["text"].str.len()
    result["token_count"] = result["text"].str.split().str.len()
    result["question_count"] = result["text"].str.count(r"\?")
    result["exclamation_count"] = result["text"].str.count("!")
    result["emoji_count"] = result["text"].map(
        lambda text: len(re.findall(r"[^\w\s\u0600-\u06FF.,!?؛،؟]", text))
    )
    result["characters_per_token"] = (
        result["character_count"] / result["token_count"].clip(lower=1)
    )
    return result

engineered = add_surface_features(responses)
engineered

Surface features are easy to reproduce and inspect, but they are not automatically valid
proxies for linguistic competence. Document their motivation and check whether they mainly
encode source, prompt, device, or demographic differences.

## 2. Remove forbidden columns before selection

In [ ]:
target = "needs_review"
forbidden = ["response_id", "annotated_errors", target]
candidate_features = engineered.drop(columns=forbidden + ["text"])

assert target not in candidate_features.columns
print("Candidate features:", candidate_features.columns.tolist())

`response_id` is an identifier. `annotated_errors` is a post-outcome measurement closely
tied to the target. An automatic selector might rank either one highly; that would not make
it a legitimate feature.

## 3. Remove constant features

In [ ]:
X_train = candidate_features.iloc[:4].copy()
X_validation = candidate_features.iloc[4:].copy()
y_train = responses.loc[:3, target]

X_train["collection_year"] = 2026
X_validation["collection_year"] = 2026

variance_filter = VarianceThreshold(threshold=0.0)
X_train_variable = variance_filter.fit_transform(X_train)
X_validation_variable = variance_filter.transform(X_validation)

selected_names = X_train.columns[variance_filter.get_support()].tolist()
print("Remaining after variance filtering:", selected_names)

## 4. Supervised selection learns from the target

In [ ]:
selector = SelectKBest(
    score_func=partial(mutual_info_classif, random_state=42),
    k=3,
)
selector.fit(X_train_variable, y_train)

X_train_selected = selector.transform(X_train_variable)
X_validation_selected = selector.transform(X_validation_variable)
final_names = np.array(selected_names)[selector.get_support()].tolist()

print("Selected training shape:", X_train_selected.shape)
print("Selected validation shape:", X_validation_selected.shape)
print("Selected names:", final_names)

Because `SelectKBest` uses `y_train`, it must be fitted inside the training workflow. During
cross-validation it belongs inside the pipeline so that each fold learns its own selection.
Scores can be unstable in small datasets; report the procedure rather than treating the
selected set as linguistic truth.

## A practical selection order

1. Remove target-derived, unavailable, sensitive-without-justification, and identifier columns.
2. Remove duplicates and constant or near-constant features.
3. Use domain knowledge to retain interpretable measurements.
4. Control high dimensionality with documented thresholds or selectors.
5. Fit target-aware selection only on training folds.
6. Test stability across splits, seeds, or bootstrap samples.

## Tiny checkpoint

Decide whether each candidate is valid for predicting review status: response ID, raw text,
token count, final adjudicated error count, task type, first language, and annotator's final
confidence. Explain the prediction-time and ethical assumptions.